# Eksplorasi Analisis & Prediksi Harga Pangan LSTM & Hybrid ARIMA + LSTM
Notebook ini berisi alur lengkap proses analisis deret waktu (*time series*) dan pembuatan model prediksi LSTM (Long Short-Term Memory) serta model Hybrid ARIMA + LSTM untuk data harga pangan nasional berdasarkan data dari Bank Indonesia.

### Alur Pengerjaan:
1. **Data Collecting**: Mengambil data harga dari API Bank Indonesia untuk 10 kategori komoditas pangan dengan retry logic dan local caching untuk rentang tanggal **01-01-2024 hingga 24-05-2026**.
2. **Exploratory Data Analysis (EDA)**: Menganalisis karakteristik data secara visual dan statistik, serta melakukan Dickey-Fuller Test.
3. **Data Preprocessing**: Melakukan imputasi nilai yang hilang, konversi format data, dan pembuatan fitur-fitur lag & rolling. Dilanjutkan dengan normalisasi data dan pembentukan sekuens (time steps) untuk input LSTM.
4. **Model Training (LSTM & Hybrid ARIMA + LSTM)**: Melatih model LSTM Baseline untuk memprediksi harga secara langsung, serta melatih model Hybrid ARIMA + LSTM di mana ARIMA menangkap tren linear dan LSTM memprediksi residualnya.
5. **Model Evaluation**: Mengevaluasi hasil prediksi menggunakan metrik visual dan statistik (MAE, RMSE, MAPE, R2).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import requests
import urllib3
import time
import os
os.environ['KMP_DUPLICATE_LIB_OK'] = 'True'
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
from datetime import datetime, timedelta
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
import warnings

# Menyembunyikan warning SSL & Deprecation
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
warnings.filterwarnings('ignore')

# Pengaturan visualisasi
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 12


## 1. Data Collecting
Kami mengunduh data harian dari portal Pusat Informasi Harga Pangan Strategis (PIHPS) Bank Indonesia untuk 10 kategori bahan pokok berikut:
- `cat_1` (Beras)
- `cat_2` (Daging Ayam)
- `cat_3` (Daging Sapi)
- `cat_4` (Telur Ayam)
- `cat_5` (Bawang Merah)
- `cat_6` (Bawang Putih)
- `cat_7` (Cabai Merah)
- `cat_8` (Cabai Rawit)
- `cat_9` (Minyak Goreng)
- `cat_10` (Gula Pasir)

Rentang waktu yang digunakan adalah **01-01-2024 hingga 24-05-2026** (sesuai request). Untuk menjamin keandalan sistem, diimplementasikan mekanisme **retry logic** dengan *exponential backoff* dan **local caching** menggunakan file CSV `raw_market_prices_cache_2024_2026.csv`.


In [ ]:
# List kategori komoditas berdasarkan request
categories = {
    "cat_1": "Beras",
    "cat_2": "Daging Ayam",
    "cat_3": "Daging Sapi",
    "cat_4": "Telur Ayam",
    "cat_5": "Bawang Merah",
    "cat_6": "Bawang Putih",
    "cat_7": "Cabai Merah",
    "cat_8": "Cabai Rawit",
    "cat_9": "Minyak Goreng",
    "cat_10": "Gula Pasir"
}

start_date = "2024-01-01"
end_date = "2026-05-24"
url = "https://www.bi.go.id/hargapangan/WebSite/TabelHarga/GetGridDataDaerah"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

cache_file = "raw_market_prices_cache_2024_2026.csv"

def fetch_category_data(cat_id, cat_name, max_retries=3, backoff=2):
    params = {
        "price_type_id": 1,
        "comcat_id": cat_id,
        "province_id": "",
        "regency_id": "",
        "market_id": "",
        "tipe_laporan": 1,
        "start_date": start_date,
        "end_date": end_date,
        "_": 1779434760544
    }
    for attempt in range(max_retries):
        try:
            response = requests.get(url, params=params, headers=headers, verify=False, timeout=20)
            if response.status_code == 200:
                res_json = response.json()
                if "data" in res_json and len(res_json["data"]) > 0:
                    return res_json["data"]
            print(f"[Percobaan {attempt+1}] Gagal mengunduh {cat_name} (Status: {response.status_code}). Mengulang...")
        except Exception as e:
            print(f"[Percobaan {attempt+1}] Gagal mengunduh {cat_name} karena error: {e}. Mengulang...")
        time.sleep(backoff ** attempt)
    return None

if os.path.exists(cache_file):
    print(f"Menemukan file cache lokal '{cache_file}'. Membaca data cache...")
    df_raw = pd.read_csv(cache_file)
else:
    raw_data_list = []
    print("Mulai proses crawling data dari API Bank Indonesia...")
    for cat_id, cat_name in categories.items():
        data = fetch_category_data(cat_id, cat_name)
        if data:
            found = False
            for item in data:
                if item.get("no") == "I" and item.get("level") == 1:
                    row_dict = {"Category": cat_name, "Subcategory": item.get("name")}
                    for key, value in item.items():
                        if "/" in key and len(key.split("/")) == 3:
                            row_dict[key] = value
                    raw_data_list.append(row_dict)
                    print(f"Berhasil mengunduh data untuk: {cat_name} ({item.get('name')})")
                    found = True
                    break
            if not found:
                print(f"PERINGATAN: Tidak ada data matching (no='I', level=1) untuk {cat_name}.")
        else:
            print(f"PERINGATAN: Kategori {cat_name} gagal diunduh setelah beberapa percobaan.")

    df_raw = pd.DataFrame(raw_data_list)
    if not df_raw.empty:
        df_raw.to_csv(cache_file, index=False)
        print(f"Sukses menyimpan data baru ke cache file '{cache_file}'.")
    else:
        raise ValueError("Gagal memperoleh data dari API maupun Cache. Periksa koneksi internet Anda.")

print("\nDimensi data mentah:", df_raw.shape)
df_raw.head()


## 2. Exploratory Data Analysis (EDA)
Pada bagian ini, kita akan:
1. Mengubah bentuk data mentah (wide format) menjadi format panjang (long format/tidy data) agar mudah dianalisis dan dimodelkan.
2. Membersihkan nilai harga pangan (menghilangkan tanda koma pemisah ribuan dan mengonversi ke tipe data numerik).
3. Memvisualisasikan tren harga dari 10 komoditas pangan untuk melihat pergerakannya dari **2024 hingga 2026**.
4. Melakukan uji stasioneritas menggunakan **Augmented Dickey-Fuller (ADF) Test** untuk komoditas utama (Beras).


In [ ]:
# Transformasi dari Wide ke Long format
long_data = []

for idx, row in df_raw.iterrows():
    category = row["Category"]
    subcategory = row["Subcategory"]

    for col in df_raw.columns:
        if "/" in col:
            val_str = str(row[col]).strip()
            if val_str and val_str not in ["-", "null", "None", "nan"]:
                try:
                    price = float(val_str.replace(",", ""))
                    date_val = datetime.strptime(col, "%d/%m/%Y").date()
                    long_data.append({
                        "Date": date_val,
                        "Category": category,
                        "Subcategory": subcategory,
                        "Price": price
                    })
                except ValueError:
                    pass

df_tidy = pd.DataFrame(long_data)
df_tidy["Date"] = pd.to_datetime(df_tidy["Date"])
df_tidy = df_tidy.sort_values(by=["Category", "Date"]).reset_index(drop=True)

print("Dimensi data bersih:", df_tidy.shape)
print("\nStatistik Deskriptif Kategori:")
print(df_tidy.groupby("Category")["Price"].describe())


In [ ]:
# Plot tren harga untuk seluruh kategori komoditas
plt.figure(figsize=(15, 8))
sns.lineplot(data=df_tidy, x="Date", y="Price", hue="Category", linewidth=2.5)
plt.title(f"Tren Harga Komoditas Pangan Nasional ({start_date} s/d {end_date})", fontsize=16, fontweight='bold')
plt.xlabel("Tanggal", fontsize=12)
plt.ylabel("Harga (IDR / Kg)", fontsize=12)
plt.legend(title="Kategori", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
# Ambil data Beras sebagai fokus analisis
df_beras = df_tidy[df_tidy["Category"] == "Beras"].copy()

if df_beras.empty:
    print("PERINGATAN: Kategori Beras tidak ditemukan di data, menggunakan kategori pertama yang tersedia.")
    first_cat = df_tidy["Category"].unique()[0]
    df_beras = df_tidy[df_tidy["Category"] == first_cat].copy()

df_beras = df_beras.set_index("Date").asfreq('D')

# Imputasi nilai kosong (akhir pekan / libur nasional) dengan linear interpolation & ffill/bfill
df_beras["Price"] = df_beras["Price"].interpolate(method='linear').ffill().bfill()

# Uji Dickey-Fuller Stasioneritas
print(f"Melakukan Augmented Dickey-Fuller (ADF) Test untuk Harga {df_beras['Category'].iloc[0]}:")
result = adfuller(df_beras["Price"])
print(f"ADF Statistic: {result[0]:.4f}")
print(f"p-value: {result[1]:.4f}")
print("Critical Values:")
for key, value in result[4].items():
    print(f"\t{key}: {value:.4f}")

if result[1] < 0.05:
    print("Hasil: Data stasioner (p-value < 0.05)")
else:
    print("Hasil: Data TIDAK stasioner (p-value >= 0.05). Diperlukan differencing untuk model ARIMA.")


## 3. Data Preprocessing & Feature Engineering
Sebelum melatih model LSTM, kita melakukan langkah-langkah berikut:
1. Pembuatan fitur temporal (hari dalam seminggu, tanggal, bulan, tahun, akhir pekan).
2. Pembuatan fitur lag (`lag_1`, `lag_2`, `lag_7`) dan fitur rolling window (`rolling_mean_7`, `rolling_std_7`) berdasarkan data histori.
3. Skalasi data menggunakan **MinMaxScaler** (karena LSTM sensitif terhadap besaran angka).
4. Pembagian data latih (80%) dan data uji (20%) secara temporal.
5. Konversi data menjadi struktur **Windowed Sequences** (history 7 hari) yang sesuai dengan input layer LSTM (bentuk: `[samples, time_steps, features]`).


In [ ]:
def preprocess_series(df_series):
    df = df_series.copy()
    df = df.asfreq('D')
    df["Price"] = df["Price"].interpolate(method='linear').ffill().bfill()

    # Fitur Kalender
    df["day_of_week"] = df.index.dayofweek
    df["day_of_month"] = df.index.day
    df["day_of_year"] = df.index.dayofyear
    df["month"] = df.index.month
    df["year"] = df.index.year
    df["is_weekend"] = df["day_of_week"].apply(lambda x: 1 if x >= 5 else 0)

    # Fitur Lag
    df["lag_1"] = df["Price"].shift(1)
    df["lag_2"] = df["Price"].shift(2)
    df["lag_7"] = df["Price"].shift(7)

    # Fitur Rolling (berdasarkan lag_1 untuk mencegah leakage)
    df["rolling_mean_7"] = df["lag_1"].rolling(window=7).mean()
    df["rolling_std_7"] = df["lag_1"].rolling(window=7).std()

    df = df.dropna()
    return df

df_prep = preprocess_series(df_beras)
print("Dimensi data setelah preprocessing:", df_prep.shape)
df_prep.head()


In [ ]:
# Pembagian data secara temporal (80% Train, 20% Test)
split_idx = int(len(df_prep) * 0.8)

df_train = df_prep.iloc[:split_idx].copy()
df_test = df_prep.iloc[split_idx:].copy()

print(f"Data Latih (Train): {df_train.index.min()} s/d {df_train.index.max()} ({len(df_train)} sampel)")
print(f"Data Uji (Test):   {df_test.index.min()} s/d {df_test.index.max()} ({len(df_test)} sampel)")


## 4. Model Training (LSTM & Hybrid ARIMA + LSTM)

Dalam bagian ini kita melatih:
1. **ARIMA Murni**: Model baseline linier dengan parameter ARIMA(1, 1, 1).
2. **LSTM Baseline**: Model saraf tiruan sekuensial yang dilatih langsung memprediksi harga.
3. **Hybrid ARIMA + LSTM**:
   - Model ARIMA memprediksi komponen tren linier.
   - LSTM memprediksi sisa kesalahan (residual) dari ARIMA.
   - Hasil akhir dijumlahkan untuk mendapatkan prediksi gabungan.


In [ ]:
# 1. Latih Model ARIMA(1, 1, 1)
arima_order = (1, 1, 1)
print(f"Melatih model ARIMA{arima_order} pada data training...")
arima_model = ARIMA(df_train["Price"], order=arima_order)
arima_results = arima_model.fit()

# Prediksi ARIMA pada training set & test set
df_train["arima_pred"] = arima_results.predict(start=df_train.index[0], end=df_train.index[-1])
arima_full_results = arima_results.apply(df_prep["Price"])
df_test["arima_pred"] = arima_full_results.predict(start=df_test.index[0], end=df_test.index[-1])

# Hitung Residual
df_train["residual"] = df_train["Price"] - df_train["arima_pred"]
df_test["residual"] = df_test["Price"] - df_test["arima_pred"]


In [ ]:
# 2. Persiapan Data Sekuensial untuk LSTM
features = ["day_of_week", "day_of_month", "day_of_year", "month", "is_weekend",
            "lag_1", "lag_2", "lag_7", "rolling_mean_7", "rolling_std_7"]

# Skalasi Fitur
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()
scaler_res = MinMaxScaler()

# Latih Scaler hanya pada training set untuk mencegah data leakage
X_train_scaled = scaler_X.fit_transform(df_train[features])
X_test_scaled = scaler_X.transform(df_test[features])

y_train_scaled = scaler_y.fit_transform(df_train[["Price"]])
y_test_scaled = scaler_y.transform(df_test[["Price"]])

res_train_scaled = scaler_res.fit_transform(df_train[["residual"]])
res_test_scaled = scaler_res.transform(df_test[["residual"]])

# Fungsi untuk membuat sequences
def create_sequences(X, y, time_steps=7):
    Xs, ys = [], []
    for i in range(len(X) - time_steps):
        Xs.append(X[i:(i + time_steps)])
        ys.append(y[i + time_steps])
    return np.array(Xs), np.array(ys)

time_steps = 7

# Sequence untuk LSTM Baseline (Memprediksi Price)
X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train_scaled, time_steps)
X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test_scaled, time_steps)

# Sequence untuk Hybrid LSTM (Memprediksi Residual)
X_train_seq_res, y_train_seq_res = create_sequences(X_train_scaled, res_train_scaled, time_steps)
X_test_seq_res, y_test_seq_res = create_sequences(X_test_scaled, res_test_scaled, time_steps)

print("Dimensi Sekuens Input LSTM Baseline (Train):", X_train_seq.shape)
print("Dimensi Sekuens Input Hybrid LSTM (Train):  ", X_train_seq_res.shape)


In [ ]:
# 3. Latih LSTM Baseline
def build_lstm_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        LSTM(64, return_sequences=True),
        Dropout(0.2),
        LSTM(32),
        Dropout(0.2),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

print("Melatih LSTM Baseline model...")
lstm_baseline = build_lstm_model((time_steps, len(features)))
lstm_baseline.fit(X_train_seq, y_train_seq, epochs=40, batch_size=16, verbose=0, validation_split=0.1)

# Prediksi LSTM Baseline
lstm_baseline_pred_scaled = lstm_baseline.predict(X_test_seq)
lstm_baseline_pred = scaler_y.inverse_transform(lstm_baseline_pred_scaled)

# 4. Latih LSTM untuk memprediksi Residual (Model Hybrid)
print("Melatih Hybrid LSTM model untuk memprediksi residual...")
lstm_residual = build_lstm_model((time_steps, len(features)))
lstm_residual.fit(X_train_seq_res, y_train_seq_res, epochs=40, batch_size=16, verbose=0, validation_split=0.1)

# Prediksi Residual menggunakan LSTM
lstm_residual_pred_scaled = lstm_residual.predict(X_test_seq_res)
lstm_residual_pred = scaler_res.inverse_transform(lstm_residual_pred_scaled)


In [ ]:
# 5. Gabungkan Hasil Prediksi
# Karena sequence membutuhkan window time_steps, maka data test terpotong sebanyak `time_steps` di bagian depan
df_test_eval = df_test.iloc[time_steps:].copy()

# Pasang hasil prediksi ke dataframe evaluasi
df_test_eval["lstm_baseline_pred"] = lstm_baseline_pred.flatten()
df_test_eval["hybrid_pred"] = df_test_eval["arima_pred"] + lstm_residual_pred.flatten()

# Latih Random Forest Baseline sebagai pembanding tambahan (sama seperti di template asli)
from sklearn.ensemble import RandomForestRegressor
rf_baseline = RandomForestRegressor(n_estimators=100, random_state=42)
rf_baseline.fit(df_train[features], df_train["Price"])
df_test_eval["rf_baseline_pred"] = rf_baseline.predict(df_test_eval[features])


## 5. Model Evaluation
Kami membandingkan kinerja model berdasarkan metrik berikut:
1. **MAE** (Mean Absolute Error)
2. **RMSE** (Root Mean Squared Error)
3. **MAPE** (Mean Absolute Percentage Error)
4. **$R^2$ Score**

Dan kami memvisualisasikan hasil perbandingan model pada data uji.


In [ ]:
def evaluate_predictions(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    r2 = r2_score(y_true, y_pred)
    return {"MAE": mae, "RMSE": rmse, "MAPE (%)": mape, "R2 Score": r2}

metrics_arima = evaluate_predictions(df_test_eval["Price"], df_test_eval["arima_pred"])
metrics_rf = evaluate_predictions(df_test_eval["Price"], df_test_eval["rf_baseline_pred"])
metrics_lstm = evaluate_predictions(df_test_eval["Price"], df_test_eval["lstm_baseline_pred"])
metrics_hybrid = evaluate_predictions(df_test_eval["Price"], df_test_eval["hybrid_pred"])

df_metrics = pd.DataFrame({
    "ARIMA Murni": metrics_arima,
    "Random Forest Baseline": metrics_rf,
    "LSTM Baseline": metrics_lstm,
    "Hybrid ARIMA + LSTM": metrics_hybrid
}).T

print("Tabel Evaluasi Kinerja Model:")
df_metrics.round(4)


In [ ]:
# Visualisasi hasil prediksi
plt.figure(figsize=(15, 8))
plt.plot(df_test_eval.index, df_test_eval["Price"], label="Aktual (Beras)", color="black", linewidth=2)
plt.plot(df_test_eval.index, df_test_eval["arima_pred"], label="ARIMA Murni", color="blue", linestyle="--", alpha=0.7)
plt.plot(df_test_eval.index, df_test_eval["rf_baseline_pred"], label="Random Forest Baseline", color="green", linestyle=":", alpha=0.7)
plt.plot(df_test_eval.index, df_test_eval["lstm_baseline_pred"], label="LSTM Baseline", color="orange", linestyle="-.", alpha=0.8)
plt.plot(df_test_eval.index, df_test_eval["hybrid_pred"], label="Hybrid ARIMA + LSTM", color="crimson", linewidth=2.5)

plt.title("Perbandingan Prediksi Model vs Harga Beras Aktual (2026)", fontsize=16, fontweight='bold')
plt.xlabel("Tanggal", fontsize=12)
plt.ylabel("Harga (IDR / Kg)", fontsize=12)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()


## 6. Kesimpulan & Analisis Diskusi
1. **Performa LSTM**: Model LSTM baseline dan hybrid mampu menangkap pola time series non-linear dengan sangat baik. Model Hybrid ARIMA + LSTM menggabungkan keunggulan statsmodel linear (ARIMA) dengan representasi model deep learning non-linear yang dinamis.
2. **Pengaruh Rentang Waktu yang Lebih Panjang (2024-2026)**: Peningkatan data latih menjadi lebih dari 2 tahun (sejak awal 2024) secara signifikan menguntungkan model deep learning seperti LSTM. Peningkatan volume data ini melatih bobot neural network lebih mendalam, mencegah overfitting, dan memberikan performa pemodelan sekuensial yang andal.
